# 12장 웹부록 | 염기 서열 다루기와 표적 찾기

「영화로 보는 AI 리터러시」 12장 'AI가 읽고 쓰는 생명'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 짧은 염기 서열을 입력해 **상보 가닥**을 만듭니다.
2. **유전 부호** 표로 염기 서열을 아미노산 서열로 번역하고, 글자 하나가 바뀌면 무엇이 달라지는지 봅니다.
3. 긴 서열에서 20자짜리 **표적**과 '한두 글자만 다른 자리'를 모두 찾아 셉니다.
4. 서열이 길어질수록 비슷한 자리가 얼마나 빨리 늘어나는지 그래프로 확인합니다.

책 3-1절(DNA와 유전체), 3-3절(크리스퍼와 표적 이탈), 실습 2(종이 유전자 편집)를 함께 보세요.

> **안전 안내**: 이 실습의 서열은 모두 컴퓨터가 무작위로 만들었거나 연습용으로 지어낸 **가짜 서열**입니다. 실제 생물의 유전자는 쓰지 않습니다. 이 실습은 '글자를 찾고 세는 방법'을 배우는 것이지, 무언가를 설계하거나 만드는 방법이 아닙니다. 책 4-3절에서 말한 것처럼, 이 책은 무엇이 가능해졌는지는 말하되 만드는 방법은 다루지 않습니다.

## 1단계. 상보 가닥 만들기

DNA의 네 글자는 짝이 정해져 있습니다. **A는 T와, G는 C와** 마주 봅니다. 그래서 한쪽 가닥만 알면 반대쪽 가닥을 자동으로 알 수 있습니다.

👉 **바꿔 볼 곳**: `MY_SEQ`에 A, T, G, C로 된 아무 서열이나 넣어 보세요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb
from IPython.display import display

MY_SEQ = "ATGGCTAAAGAAGCTTGGATTTCTCATTAA"   # ← 이 서열을 바꿔 보세요 (A, T, G, C만)

PAIR = {"A": "T", "T": "A", "G": "C", "C": "G"}

def complement(seq):
    return "".join(PAIR[b] for b in seq.upper())

seq = MY_SEQ.upper()
print("내 가닥  :", seq)
print("          " + "|" * len(seq))
print("상보 가닥:", complement(seq))
print()
print("참고: 실제 두 가닥은 서로 반대 방향으로 놓여 있어서, 상보 가닥을 거꾸로 읽은 것도 자주 씁니다.")
print("거꾸로 읽은 상보 가닥:", complement(seq)[::-1])

## 2단계. 유전 부호로 번역하기

책 3-1절에서 본 대로 **세 글자가 아미노산 하나**를 가리킵니다. 이 세 글자 묶음을 코돈이라고 합니다. 64가지 코돈이 아미노산 20가지와 '멈춤' 신호를 나눠 맡습니다. 이 규칙이 **유전 부호**이며, 생물학 교과서에 실린 표준 표를 그대로 씁니다.

아미노산은 알파벳 한 글자로 줄여 씁니다(예: M = 메싸이오닌, A = 알라닌). `*`는 '여기서 멈춤'입니다.

1단계의 연습용 서열을 번역해 보세요. 숨겨 둔 낱말이 나옵니다.

In [ ]:
BASES = "TCAG"
AMINO = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"   # 표준 유전 부호표
CODON_TABLE = {a + b + c: AMINO[16*i + 4*j + k]
               for i, a in enumerate(BASES) for j, b in enumerate(BASES) for k, c in enumerate(BASES)}

def translate(seq):
    """세 글자씩 끊어 아미노산 한 글자로 바꿉니다. 멈춤 신호(*)가 나오면 멈춥니다."""
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        aa = CODON_TABLE[seq[i:i+3]]
        protein += aa
        if aa == "*":
            break
    return protein

codons = [seq[i:i+3] for i in range(0, len(seq) - 2, 3)]
print("세 글자씩 끊기:", " ".join(codons))
print("아미노산      :", "   ".join(CODON_TABLE[c] for c in codons))
print("번역 결과     :", translate(seq))

# 64칸 유전 부호표를 표로 보기
table = pd.DataFrame([[f"{a+b+c}={CODON_TABLE[a+b+c]}" for b in BASES] for a in BASES for c in BASES],
                     index=[f"{a}_{c}" for a in BASES for c in BASES], columns=[f"_{b}_" for b in BASES])
print("\n표준 유전 부호표 (행: 첫째·셋째 글자, 열: 가운데 글자)")
display(table)

### 글자 하나가 바뀌면

책의 '심화 상식 | 돌연변이' 상자에서 본 것처럼, 글자 하나가 바뀌어도 같은 아미노산이 될 때가 많습니다. 하지만 어떤 한 글자는 아미노산을 바꾸거나, 중간에 '멈춤'을 만들어 버립니다.

👉 **바꿔 볼 곳**: `POSITION`(몇 번째 글자, 1부터)과 `NEW_BASE`를 바꿔 보세요.

In [ ]:
POSITION = 6     # ← 이 숫자를 바꿔 보세요 (1부터 셉니다)
NEW_BASE = "A"   # ← 이 글자를 바꿔 보세요 (A, T, G, C)

mutant = seq[:POSITION-1] + NEW_BASE + seq[POSITION:]
before, after = translate(seq), translate(mutant)
print("원래 서열 :", seq, "→", before)
print("바뀐 서열 :", mutant, "→", after)
print("          ", " " * (POSITION - 1) + "^")
if before == after:
    print("결과: 아미노산이 그대로입니다. 유전 부호의 '여유' 덕분입니다.")
elif after.endswith("*") and len(after) < len(before):
    print("결과: 중간에 멈춤 신호가 생겨 단백질이 짧게 끊겼습니다.")
else:
    print("결과: 아미노산이 바뀌었습니다.")

# 모든 자리를 한 글자씩 바꿔 보고 결과를 세어 봅니다
kinds = {"same": 0, "changed": 0, "early stop": 0}
for p in range(len(seq) - 3):          # 마지막 멈춤 코돈은 빼고
    for b in "ATGC":
        if b == seq[p]: continue
        t = translate(seq[:p] + b + seq[p+1:])
        if t == before: kinds["same"] += 1
        elif t.endswith("*") and len(t) < len(before): kinds["early stop"] += 1
        else: kinds["changed"] += 1
print("\n가능한 한 글자 돌연변이 전체:", kinds)
plt.figure(figsize=(5, 3)); plt.bar(kinds.keys(), kinds.values(), color=["tab:green", "tab:orange", "tab:red"])
plt.title("Every possible single-letter change"); plt.ylabel("count"); plt.show()

## 3단계. 긴 서열에서 표적 찾기

이제 책의 실습 2(종이 유전자 편집)를 컴퓨터로 크게 해 봅니다.

- 컴퓨터가 A, T, G, C를 무작위로 골라 **10만 자짜리 가짜 유전체**를 만듭니다.
- 무작위 **20자짜리 표적**을 하나 정해 유전체 어딘가에 한 번 넣습니다. 크리스퍼의 안내 RNA가 찾아가는 '스무 자 남짓'과 같은 길이입니다.
- 그리고 표적과 **한 글자만 다른** 서열, **두 글자만 다른** 서열을 몰래 몇 군데 숨겨 둡니다.

컴퓨터는 종이띠 위에 쪽지를 대고 한 칸씩 미는 일을 10만 번 합니다. 자리마다 '몇 글자가 다른지' 셉니다.

👉 **바꿔 볼 곳**: `GENOME_LENGTH`와 숨길 개수를 바꿔 보세요.

In [ ]:
GENOME_LENGTH = 100_000   # ← 이 숫자를 바꿔 보세요 (가짜 유전체 길이)
HIDE_1_OFF = 2            # ← 한 글자만 다른 자리를 몇 군데 숨길지
HIDE_2_OFF = 3            # ← 두 글자만 다른 자리를 몇 군데 숨길지
TARGET_LEN = 20

rng = np.random.default_rng(12)
genome = rng.choice(list("ATGC"), GENOME_LENGTH)
target = "".join(rng.choice(list("ATGC"), TARGET_LEN))

def mutate(s, n, rng):
    s = list(s)
    for p in rng.choice(len(s), n, replace=False):
        s[p] = rng.choice([b for b in "ATGC" if b != s[p]])
    return "".join(s)

spots = rng.choice(np.arange(1, GENOME_LENGTH // 1000) * 1000, 1 + HIDE_1_OFF + HIDE_2_OFF, replace=False)
hidden = [target] + [mutate(target, 1, rng) for _ in range(HIDE_1_OFF)] + [mutate(target, 2, rng) for _ in range(HIDE_2_OFF)]
for pos, piece in zip(spots, hidden):
    genome[pos:pos + TARGET_LEN] = list(piece)
genome = "".join(genome)
print("가짜 유전체 앞부분:", genome[:60], "...")
print("찾을 표적(20자)   :", target)

def scan(genome, target, max_diff=2):
    """유전체의 모든 자리에 표적을 대 보고, 다른 글자 수가 max_diff 이하인 자리를 돌려줍니다."""
    g = np.frombuffer(genome.encode(), dtype=np.uint8)
    t = np.frombuffer(target.encode(), dtype=np.uint8)
    windows = np.lib.stride_tricks.sliding_window_view(g, len(t))   # 쪽지를 한 칸씩 민 모든 자리
    diff = (windows != t).sum(axis=1)
    where = np.where(diff <= max_diff)[0]
    return [(int(p), int(diff[p])) for p in where]

found = scan(genome, target)
rows = []
for pos, d in found:
    piece = genome[pos:pos + TARGET_LEN]
    marks = "".join("^" if a != b else "." for a, b in zip(piece, target))
    rows.append({"자리(위치)": pos, "다른 글자 수": d, "찾은 서열": piece, "다른 곳(^)": marks})
print(f"\n{GENOME_LENGTH:,}자를 {GENOME_LENGTH - TARGET_LEN + 1:,}번 밀어 보았습니다.")
display(pd.DataFrame(rows).sort_values("다른 글자 수"))

**읽는 법**: '다른 글자 수'가 0인 자리가 진짜 표적입니다. 1이나 2인 자리가 **표적 이탈**(off-target)이 생길 수 있는 후보입니다. 가위가 이런 자리를 헷갈려 자르면 멀쩡한 곳이 망가집니다. 그래서 연구자들은 표적 후보를 고를 때 유전체 전체에서 이런 비슷한 자리를 먼저 찾아 점검합니다.

## 4단계. 서열이 길어지면 비슷한 자리는 얼마나 늘어날까

이번에는 아무것도 숨기지 않은 순수한 무작위 서열에서, '우연히' 생기는 비슷한 자리를 셉니다.

20자짜리 표적은 우연히 비슷한 자리가 생기기 매우 어렵습니다. 그래서 먼저 책 실습 2의 쪽지처럼 **8자짜리 표적**(`ATGCCTAG`)으로, '똑같은 자리'와 '한 글자만 다른 자리'를 셉니다. 서열 길이를 1천 자에서 100만 자까지 늘려 가며 직접 세고, 계산으로 얻은 기댓값과 비교합니다.

In [ ]:
SHORT_TARGET = "ATGCCTAG"   # ← 책 실습 2의 쪽지입니다. 바꿔 보세요
lengths = [1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000]

def expected(N, k, max_diff):
    """무작위 서열이라면 '다른 글자가 max_diff 이하인 자리'가 평균 몇 개 나올지 계산합니다."""
    p = sum(comb(k, i) * 3**i for i in range(max_diff + 1)) / 4**k
    return (N - k + 1) * p

rng = np.random.default_rng(0)
rows = []
for N in lengths:
    g = "".join(rng.choice(list("ATGC"), N))
    hits = scan(g, SHORT_TARGET, max_diff=1)
    exact = sum(1 for _, d in hits if d == 0); one = sum(1 for _, d in hits if d == 1)
    rows.append({"서열 길이": N, "똑같은 자리": exact, "한 글자만 다른 자리": one,
                 "계산한 기댓값(0~1글자 차이)": round(expected(N, len(SHORT_TARGET), 1), 1)})
growth = pd.DataFrame(rows)
display(growth)

plt.figure(figsize=(6.5, 4))
plt.loglog(growth["서열 길이"], growth["똑같은 자리"] + growth["한 글자만 다른 자리"], "o", label="counted (0-1 letter different)")
plt.loglog(lengths, [expected(N, len(SHORT_TARGET), 1) for N in lengths], "-", label="expected by calculation")
plt.loglog(lengths, [max(expected(N, len(SHORT_TARGET), 0), 1e-3) for N in lengths], "--", label="expected, exact match only")
plt.xlabel("sequence length (letters)"); plt.ylabel("number of similar spots")
plt.title(f"Similar spots for an {len(SHORT_TARGET)}-letter target"); plt.legend(); plt.show()

**읽는 법**: 두 축이 모두 10배씩 늘어나는 눈금입니다. 서열이 10배 길어지면 비슷한 자리도 거의 정확히 10배 늘어납니다. 그리고 '똑같은 자리'보다 '한 글자만 다른 자리'가 훨씬 많습니다. 한 글자가 다를 수 있는 방법이 8자리 × 3글자 = 24가지나 되기 때문입니다.

### 사람 유전체 크기(30억 자)라면?

30억 자를 직접 만들어 세기에는 코랩이 너무 느립니다. 그래서 같은 계산식으로 기댓값만 구합니다. 표적 길이를 8자에서 20자까지 늘리면 어떻게 될까요? 책 실습 2의 '생각해 볼 것'에 대한 답을 숫자로 확인해 보세요.

(무작위 서열이라고 가정한 값이고, 한쪽 가닥만 센 값입니다. 실제 유전체에는 비슷한 서열이 되풀이되는 구간이 많아서 비슷한 자리가 이보다 많이 나옵니다.)

In [ ]:
HUMAN = 3_000_000_000

def nice(x):
    if x >= 10: return f"약 {x:,.0f}곳"
    if x >= 1:  return f"약 {x:.1f}곳"
    return f"1곳보다 적음 ({x:.4f})"

rows = []
for k in [8, 12, 16, 20, 24]:
    rows.append({"표적 길이": k,
                 "똑같은 자리": nice(expected(HUMAN, k, 0)),
                 "1글자 이하 차이": nice(expected(HUMAN, k, 1)),
                 "2글자 이하 차이": nice(expected(HUMAN, k, 2))})
display(pd.DataFrame(rows))

Ns = np.logspace(3, np.log10(HUMAN), 50)
plt.figure(figsize=(6.5, 4))
for k, m in [(20, 0), (20, 1), (20, 2)]:
    plt.loglog(Ns, [expected(N, k, m) for N in Ns], label=f"20 letters, up to {m} different")
plt.axhline(1, color="gray", ls=":"); plt.axvline(HUMAN, color="gray", ls="--")
plt.text(HUMAN, 1e-6, " human genome\n (3 billion)", ha="right", fontsize=8)
plt.xlabel("sequence length (letters)"); plt.ylabel("expected similar spots")
plt.title("A 20-letter target in longer and longer sequences"); plt.legend(); plt.show()

**읽는 법**: 8자짜리 표적은 30억 자 안에서 똑같은 자리만 수만 곳이 넘습니다. 어디를 자를지 정할 수 없습니다. 20자로 늘리면 똑같은 자리는 거의 없어지지만, 그래도 '두 글자만 다른 자리'는 몇 곳쯤 남습니다. 그래서 표적 길이만으로는 부족하고, 비슷한 자리를 하나하나 점검하는 일이 필요합니다. 책에서 '정확한 가위는 어디를 자를지 잘 고르는 가위'라고 한 이유입니다.

## 더 해 보기

- 3단계의 `GENOME_LENGTH`를 `1_000_000`으로 늘려 보세요. 숨기지 않은 '우연한' 비슷한 자리가 새로 나타납니까? 4단계의 계산과 맞습니까?
- 4단계의 `SHORT_TARGET`을 `AAAAAAAA`처럼 같은 글자가 되풀이되는 서열로 바꿔 보세요. 무작위 서열에서는 결과가 비슷하지만, 실제 유전체에서는 왜 이런 서열이 특히 위험할지 생각해 보세요.

## 생각해 볼 것

1. 20자짜리 표적도 30억 자 안에서는 '한두 글자만 다른 자리'가 생길 수 있었습니다. 크리스퍼 치료를 받을 환자에게, 의사는 이 위험을 어떻게 설명해야 할까요? 책 3-3절의 '편집한 뒤 다시 읽어 확인하기'와 연결해 쓰세요.
2. 이 실습의 '찾기'는 단순한 글자 비교였습니다. 실제 연구자들은 여기에 AI를 더해 '이 자리가 정말 잘릴 확률'을 예측합니다. AI의 예측을 믿고 점검 단계를 줄여도 될까요? 6장의 '데이터 치우침'과 함께 생각해 보세요.
3. 같은 '서열 찾기' 기술이 DNA 합성 회사의 위험 서열 검사에도 쓰입니다(책 4-3절). 이처럼 같은 기술이 치료에도, 안전 점검에도, 나쁜 일에도 쓰일 수 있는 '이중 용도' 상황에서, 공개해도 되는 지식과 조심해야 할 지식의 경계는 어디라고 생각합니까?